In [1]:
import os
import copy
import logging
import h5py as h5
import numpy as np
import pandas as pd
import astropy.units as u
import matplotlib.pyplot as plt
import matplotlib

import matplotlib.cm as cm



# ------------------------------
# Plot style
# ------------------------------
matplotlib.rcParams.update({
    'font.size': 20,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 6,
    'xtick.minor.size': 3,
    'ytick.major.size': 6,
    'ytick.minor.size': 3,
    'xtick.major.width': 1,
    'ytick.major.width': 1,
    'xtick.minor.width': 0.8,
    'ytick.minor.width': 0.8,
    'xtick.top': True,
    'ytick.right': True,
    'xtick.bottom': True,
    'ytick.left': True,
    
    'text.usetex': True,  # Enable LaTeX rendering
    'font.family': 'serif',
    'font.serif': ['Computer Modern'],  # LaTeX default font
})

Zsun = 0.02  
threshold = 1.00 #threshold for MRR, mB > threshold * mA
# ------------------------------
# Load COMPAS data
# ------------------------------

with h5.File("processed_yields_COMPAS.h5", "r") as f:
    z_vals_compas = f["z_values"][()]
    Y_compas      = f["yields"][()]

    # LVK defined M1
    m1_compas     = np.maximum(f["event_properties/mass_1"][()],
                                f["event_properties/mass_2"][()])
    m2_compas     = np.minimum(f["event_properties/mass_1"][()],
                                f["event_properties/mass_2"][()])
    
    MRR_bh_compas = (f["event_properties/mass_2"][()] > threshold *
                     f["event_properties/mass_1"][()])

    Z_compas      = f["event_properties/metallicity"][()]  
    logZrel_compas  = np.log10(Z_compas / Zsun)

    # Just ensure we are grabbing M1 and M2 ZAMS. 
    zams1_compas  = np.maximum(f["event_properties/zams_mass_1"][()],
                                f["event_properties/zams_mass_2"][()])
    zams2_compas  = np.minimum(f["event_properties/zams_mass_1"][()],
                                f["event_properties/zams_mass_2"][()])
    
    delay_time_compas = f["event_properties/delay_time_in_years"][()]  

    # MASSquerade

    m1_compas_binary_assumption = f["event_properties/mass_1"][()]
    m2_compas_binary_assumption = f["event_properties/mass_2"][()]

# ------------------------------
# Load SEVN data
# ------------------------------

# with h5.File("processed_yields_SEVN.h5", "r") as f:
with h5.File("../data/yields/sevn_updated.hdf5", "r") as f:
    z_vals_sevn = f["z_values"][()]
    Y_sevn      = f["yields"][()]

    m1_sevn     = np.maximum(f["event_properties/mass_1"][()],
                              f["event_properties/mass_2"][()])
    m2_sevn     = np.minimum(f["event_properties/mass_1"][()],
                              f["event_properties/mass_2"][()])
    MRR_bh_sevn = (f["event_properties/mass_2"][()] > threshold *
                   f["event_properties/mass_1"][()])

    Z_sevn      = f["event_properties/metallicity"][()]
    logZrel_sevn  = np.log10(Z_sevn / Zsun)

    zams1_sevn  = np.maximum(f["event_properties/zams_mass_1"][()],
                              f["event_properties/zams_mass_2"][()])
    zams2_sevn  = np.minimum(f["event_properties/zams_mass_1"][()],
                                f["event_properties/zams_mass_2"][()])

    delay_time_sevn = f["event_properties/delay_time_in_years"][()]  

    # MASSquerade

    m1_sevn_binary_assumption = f["event_properties/mass_1"][()]
    m2_sevn_binary_assumption = f["event_properties/mass_2"][()]


In [ ]:
#####
# M1
#####
m1_bins=np.arange(3,81,1)
m1_centers = 0.5 * (m1_bins[:-1] + m1_bins[1:])

dRdM1_sevn = np.array([np.histogram(m1_sevn, bins=m1_bins, weights=Y_sevn[i])[0] for i in range(len(z_vals_sevn))])
dRdM1_MRR_sevn = np.array(np.histogram(m1_sevn[MRR_bh_sevn], bins=m1_bins, weights=Y_sevn[2][MRR_bh_sevn])[0])
dRdM1_nonMRR_sevn = np.array(np.histogram(m1_sevn[~MRR_bh_sevn], bins=m1_bins, weights=Y_sevn[2][~MRR_bh_sevn])[0])

dRdM1_compas = np.array([np.histogram(m1_compas, bins=m1_bins, weights=Y_compas[i])[0] for i in range(len(z_vals_compas))])
dRdM1_MRR_compas = np.array(np.histogram(m1_compas[MRR_bh_compas], bins=m1_bins, weights=Y_compas[2][MRR_bh_compas])[0])
dRdM1_nonMRR_compas = np.array(np.histogram(m1_compas[~MRR_bh_compas], bins=m1_bins, weights=Y_compas[2][~MRR_bh_compas])[0])

#####
# M2
#####

m2_bins=np.arange(3,81,1)
m2_centers = 0.5 * (m1_bins[:-1] + m1_bins[1:])
dRdM2_sevn = np.array([np.histogram(m2_sevn, bins=m2_bins, weights=Y_sevn[i])[0] for i in range(len(z_vals_sevn))])
dRdM2_MRR_sevn = np.array(np.histogram(m2_sevn[MRR_bh_sevn], bins=m2_bins, weights=Y_sevn[2][MRR_bh_sevn])[0])
dRdM2_nonMRR_sevn = np.array(np.histogram(m2_sevn[~MRR_bh_sevn], bins=m2_bins, weights=Y_sevn[2][~MRR_bh_sevn])[0])

dRdM2_compas = np.array([np.histogram(m2_compas, bins=m2_bins, weights=Y_compas[i])[0] for i in range(len(z_vals_compas))])
dRdM2_MRR_compas = np.array(np.histogram(m2_compas[MRR_bh_compas], bins=m2_bins, weights=Y_compas[2][MRR_bh_compas])[0])
dRdM2_nonMRR_compas = np.array(np.histogram(m2_compas[~MRR_bh_compas], bins=m2_bins, weights=Y_compas[2][~MRR_bh_compas])[0])

#####
# q
#####

q_bins = np.arange(0,1.05,0.05)
q_centers = 0.5 * (q_bins[:-1] + q_bins[1:])
q_sevn = m2_sevn/m1_sevn # BH q
dRdq_sevn = np.array([np.histogram(q_sevn, bins=q_bins, weights=Y_sevn[i])[0] for i in range(len(z_vals_sevn))])
dRdq_MRR_sevn = np.array(np.histogram(q_sevn[MRR_bh_sevn], bins=q_bins, weights=Y_sevn[2][MRR_bh_sevn])[0])
dRdq_nonMRR_sevn = np.array(np.histogram(q_sevn[~MRR_bh_sevn], bins=q_bins, weights=Y_sevn[2][~MRR_bh_sevn])[0])


dq = q_centers[2]-q_centers[1]
dRdq_sevn/=dq
dRdq_MRR_sevn/=dq
dRdq_nonMRR_sevn/=dq

q_compas = m2_compas/m1_compas # BH q
dRdq_compas = np.array([np.histogram(q_compas, bins=q_bins, weights=Y_compas[i])[0] for i in range(len(z_vals_compas))])
dRdq_MRR_compas = np.array(np.histogram(q_compas[MRR_bh_compas], bins=q_bins, weights=Y_compas[2][MRR_bh_compas])[0])
dRdq_nonMRR_compas = np.array(np.histogram(q_compas[~MRR_bh_compas], bins=q_bins, weights=Y_compas[2][~MRR_bh_compas])[0])

dq = q_centers[2]-q_centers[1]
dRdq_compas/=dq
dRdq_MRR_compas/=dq
dRdq_nonMRR_compas/=dq

In [ ]:
# Load secondary data

# Sadiq secondaries
from scipy.interpolate import interp1d

df_sadiq = pd.read_csv('../data/sadiq_secondaries.csv')
sadiq_x,sadiq_y = df_sadiq['x'],df_sadiq[' y']

# Load all three CSVs
df_mean = pd.read_csv('../data/sadiq_secondaries.csv')
df_upper = pd.read_csv('../data/sadiq_up.csv')
df_lower = pd.read_csv('../data/sadiq_low.csv')

# Extract data (note: remove leading space in column names if present)
x_mean = df_mean['x']
y_mean = df_mean[' y']  # if this has a leading space

x_upper = df_upper['x']
y_upper = df_upper[' y']

x_lower = df_lower['x']
y_lower = df_lower[' y']

# Interpolate upper and lower bounds to match x_mean
interp_upper = interp1d(x_upper, y_upper, bounds_error=False, fill_value='extrapolate')
interp_lower = interp1d(x_lower, y_lower, bounds_error=False, fill_value='extrapolate')

y_upper_interp = interp_upper(x_mean)
y_lower_interp = interp_lower(x_mean)

In [ ]:
# Mass ratio
data = np.load(
    '../data/mass_ratio_data.npz'
)

pp_q      = data['gwtc3_q']
pp_ppd    = data['gwtc3_ppd']
pp_lo     = data['gwtc3_lo']
pp_hi     = data['gwtc3_hi']

bs_q       = data['bs_q']
bs_q_pdfs  = data['bs_q_pdfs']

bptp_q      = data['bptp_q']
bptp_q_pdfs = data['bptp_q_pdfs']

d = np.load("../data/bspline_massdist_panel0.npz")

x = d["x"]
md = d["median"]
lo = d["lower"]
hi = d["upper"]


In [ ]:
fig,axs = plt.subplots(3,2,sharey=True, figsize=(16,14))

fig.set_constrained_layout_pads(hspace=0.0,wspace=0.0)

cmrr = "#9467BD" #purple
cnomrr= "#FF7F0E" #orange
cnomrr = "#2CA02C"

#########
# dRdM1 # 
#########
axs[0,0].plot(m1_centers,dRdM1_compas[2],c='k',lw=4,label="Total")
axs[0,0].plot(m1_centers,dRdM1_nonMRR_compas,lw=4,ls=':',c=cnomrr,label="Non-MRR")
axs[0,0].plot(m1_centers,dRdM1_MRR_compas,lw=4,c=cmrr,ls='--',label="MRR")
axs[0,0].plot(x, md, color="gray", lw=2, label="GWTC-4.0: B-Spline")
axs[0,0].fill_between(x, lo, hi, color="gray", alpha=0.3)
axs[0,0].legend(frameon=False,fontsize=18)
axs[0,0].set_yscale("log")
axs[0,0].set_xlim(3,48)
axs[0,0].set_ylim(1e-2,1e2)
axs[0,0].set_ylabel(r"$d\mathcal{R}/dM_1$ [Gpc$^{-3}$ yr$^{-1}$ M$_\odot^{-1}$]")
axs[0,0].text(5,100,"COMPAS",fontsize=30)#,fontfamily='cursive',fontweight='bold')
axs[0,0].set_xlabel(r"$M_1 \, [M_\odot]$")


axs[0,1].plot(m1_centers,dRdM1_sevn[2],c='k',lw=4,label="Total")
axs[0,1].plot(m1_centers,dRdM1_nonMRR_sevn,lw=4,c=cnomrr,ls=':',label="Non-MRR")
axs[0,1].plot(m1_centers,dRdM1_MRR_sevn,lw=4,c=cmrr,ls='--',label="MRR")
axs[0,1].plot(x, md, color="gray", lw=2, label="GWTC-4.0: B-Spline")
axs[0,1].fill_between(x, lo, hi, color="gray", alpha=0.3)
axs[0,1].legend(frameon=False,fontsize=18)
axs[0,1].set_yscale("log")
axs[0,1].set_xlim(3,48)
axs[0,1].set_ylim(1e-2,1e2)
axs[0,1].text(5,100,"SEVN",fontsize=30)#,fontfamily='cursive',fontweight='bold')
axs[0,1].set_xlabel(r"$M_1 \, [M_\odot]$")


#########
# dRdM2 # 
#########

axs[1,0].plot(m2_centers,dRdM2_compas[2],c='k',lw=4,label="Total")
axs[1,0].plot(m2_centers,dRdM2_nonMRR_compas,lw=4,c=cnomrr,ls=':',label="Non-MRR")
axs[1,0].plot(m2_centers,dRdM2_MRR_compas,lw=4,c=cmrr,ls='--',label="MRR")
axs[1,0].plot(x_mean, y_mean, color='k', linewidth=2, alpha=0.3)
axs[1,0].fill_between(x_mean, y_lower_interp, y_upper_interp, color='gray', alpha=0.3,label="Sadiq et al. 2024")
axs[1,0].legend(frameon=False,fontsize=18)
axs[1,0].set_yscale("log")
axs[1,0].set_xlim(3,48)
axs[1,0].set_ylim(1e-2,1e2)
axs[1,0].set_ylabel(r"$d\mathcal{R}/dM_2$ [Gpc$^{-3}$ yr$^{-1}$ M$_\odot^{-1}$]")
axs[1,0].set_xlabel(r"$M_2 \, [M_\odot]$")

axs[1,1].plot(m2_centers,dRdM2_sevn[2],c='k',lw=4,label="Total")
axs[1,1].plot(m2_centers,dRdM2_nonMRR_sevn,lw=4,c=cnomrr,ls=':',label="Non-MRR")
axs[1,1].plot(m2_centers,dRdM2_MRR_sevn,lw=4,c=cmrr,ls='--',label="MRR")
axs[1,1].plot(x_mean, y_mean, color='k', linewidth=2, alpha=0.3)
axs[1,1].fill_between(x_mean, y_lower_interp, y_upper_interp, color='gray', alpha=0.3,label="Sadiq et al. 2024")
axs[1,1].legend(frameon=False,fontsize=18)
axs[1,1].set_yscale("log")
axs[1,1].set_xlim(3,48)
axs[1,1].set_ylim(1e-2,1e2)
axs[1,1].set_xlabel(r"$M_2 \, [M_\odot]$")

#########
# dRdq # 
#########

axs[2,0].plot(q_centers,dRdq_compas[2],c='k',lw=4,label="Total")
axs[2,0].plot(q_centers,dRdq_nonMRR_compas,lw=4,c=cnomrr,ls=':',label="Non-MRR")
axs[2,0].plot(q_centers,dRdq_MRR_compas,lw=4,c=cmrr,ls='--',label="MRR")
axs[2,0].fill_between(
    bs_q,
    np.percentile(bs_q_pdfs, 5, axis=0),
    np.percentile(bs_q_pdfs, 95, axis=0),
    color="gray", alpha=0.3,
    label=r'B-Spline, GWTC-4.0'
)
axs[2,0].plot(bs_q, np.percentile(bs_q_pdfs, 50, axis=0),
        color="gray", lw=2)
axs[2,0].legend(frameon=False,fontsize=18)
axs[2,0].set_yscale("log")
axs[2,0].set_xlim(0.05,1)
axs[2,0].set_ylim(1e-2,1e3)
axs[2,0].set_ylabel(r"$d\mathcal{R}/dq$ [Gpc$^{-3}$ yr$^{-1}$]")
axs[2,0].set_xlabel("q")

axs[2,1].plot(q_centers,dRdq_sevn[2],c='k',lw=4,label="Total")
axs[2,1].plot(q_centers,dRdq_nonMRR_sevn,lw=4,c=cnomrr,ls=':',label="Non-MRR")
axs[2,1].plot(q_centers,dRdq_MRR_sevn,lw=4,c=cmrr,ls='--',label="MRR")
axs[2,1].fill_between(
    bs_q,
    np.percentile(bs_q_pdfs, 5, axis=0),
    np.percentile(bs_q_pdfs, 95, axis=0),
    color="gray", alpha=0.3,
    label=r'B-Spline, GWTC-4.0'
)
axs[2,1].plot(bs_q, np.percentile(bs_q_pdfs, 50, axis=0),
        color="gray", lw=2)
axs[2,1].legend(frameon=False,fontsize=18)
axs[2,1].set_yscale("log")
axs[2,1].set_xlim(0.05,1)
axs[2,1].set_ylim(1e-2,1e3)
axs[2,1].set_xlabel("q")






plt.minorticks_on()
#plt.tight_layout()
plt.subplots_adjust(
    wspace=0.05,   
    hspace=0.35,
    left=0.0,
    right=1,
)
plt.savefig("../plots/final-dRdX.pdf", bbox_inches="tight", pad_inches=0.04)
plt.show()

# Massquerade fig

In [ ]:
#####
# M1
#####

# dRdM1_sevn already made!
dRdM1_sevn_binary = np.array(np.histogram(m1_sevn_binary_assumption, bins=m1_bins, weights=Y_sevn[2])[0])
dRdM1_compas_binary = np.array(np.histogram(m1_compas_binary_assumption, bins=m1_bins, weights=Y_compas[2])[0])#  for i in range(len(z_vals_sevn))])


#####
# M2
#####

dRdM2_sevn_binary = np.array(np.histogram(m2_sevn_binary_assumption, bins=m2_bins, weights=Y_sevn[2])[0])
dRdM2_compas_binary = np.array(np.histogram(m2_compas_binary_assumption, bins=m2_bins, weights=Y_compas[2])[0])#  for i in range(len(z_vals_sevn))])


In [ ]:
fig,axs = plt.subplots(2,2,sharey=True, figsize=(16,10))

fig.set_constrained_layout_pads(hspace=0.0,wspace=0.0)

cmrr = "#9467BD" #purple
cnomrr= "#FF7F0E" #orange
cnomrr = "#2CA02C"

#########
# dRdM1 # 
#########
axs[0,0].plot(m1_centers,dRdM1_compas[2],c='k',lw=4,label=r'LVK-defined M$_1$' )
axs[0,0].plot(m1_centers,dRdM1_compas_binary,lw=4,ls='--',c="r",label=r'Binary-defined M$_1$' )

axs[0,0].plot(x, md, color="gray", lw=2, label="GWTC-4.0: B-Spline")
axs[0,0].fill_between(x, lo, hi, color="gray", alpha=0.3)
axs[0,0].legend(frameon=False,fontsize=18)
axs[0,0].set_yscale("log")
axs[0,0].set_xlim(3,48)
axs[0,0].set_ylim(1e-2,1e2)
axs[0,0].set_ylabel(r"$d\mathcal{R}/dM_1$ [Gpc$^{-3}$ yr$^{-1}$ M$_\odot^{-1}$]")
axs[0,0].text(20,30,"COMPAS",fontsize=30)#,fontfamily='cursive',fontweight='bold')
axs[0,0].set_xlabel(r"$M_1 \, [M_\odot]$")


axs[0,1].plot(m1_centers,dRdM1_sevn[2],c='k',lw=4,label=r'LVK-defined M$_1$' )
axs[0,1].plot(m1_centers,dRdM1_sevn_binary,lw=4,c="r",ls='--',label=r'Binary-defined M$_1$' )

axs[0,1].plot(x, md, color="gray", lw=2, label="GWTC-4.0: B-Spline")
axs[0,1].fill_between(x, lo, hi, color="gray", alpha=0.3)
axs[0,1].legend(frameon=False,fontsize=18)
axs[0,1].set_yscale("log")
axs[0,1].set_xlim(3,48)
axs[0,1].set_ylim(1e-2,1e2)
axs[0,1].text(20,30,"SEVN",fontsize=30)#,fontfamily='cursive',fontweight='bold')
axs[0,1].set_xlabel(r"$M_1 \, [M_\odot]$")


#########
# dRdM2 # 
#########

axs[1,0].plot(m2_centers,dRdM2_compas[2],c='k',lw=4,label=r'LVK-defined M$_2$' )
axs[1,0].plot(m2_centers,dRdM2_compas_binary,lw=4,c="r",ls='--',label=r'Binary-defined M$_2$' )

axs[1,0].plot(x_mean, y_mean, color='k', linewidth=2, alpha=0.3)
axs[1,0].fill_between(x_mean, y_lower_interp, y_upper_interp, color='gray', alpha=0.3,label="Sadiq et al. 2024")
axs[1,0].legend(frameon=False,fontsize=18)
axs[1,0].set_yscale("log")
axs[1,0].set_xlim(3,48)
axs[1,0].set_ylim(1e-2,1e2)
axs[1,0].text(20,30,"COMPAS",fontsize=30)
axs[1,0].set_ylabel(r"$d\mathcal{R}/dM_2$ [Gpc$^{-3}$ yr$^{-1}$ M$_\odot^{-1}$]")
axs[1,0].set_xlabel(r"$M_2 \, [M_\odot]$")
# plt.minorticks_on()
axs[1,1].plot(m2_centers,dRdM2_sevn[2],c='k',lw=4,label=r'LVK-defined M$_2$' )
axs[1,1].plot(m2_centers,dRdM2_sevn_binary,lw=4,c="r",ls='--',label=r'Binary-defined M$_2$' )

axs[1,1].plot(x_mean, y_mean, color='k', linewidth=2, alpha=0.3)
axs[1,1].fill_between(x_mean, y_lower_interp, y_upper_interp, color='gray', alpha=0.3,label="Sadiq et al. 2024")
axs[1,1].legend(frameon=False,fontsize=18)
axs[1,1].set_yscale("log")
axs[1,1].set_xlim(3,48)
axs[1,1].set_ylim(1e-2,1e2)
axs[1,1].text(20,30,"SEVN",fontsize=30)
axs[1,1].set_xlabel(r"$M_2 \, [M_\odot]$")
axs[0,0].minorticks_on()

# plt.minorticks_on()
axs[0,0].minorticks_on()
axs[0,1].minorticks_on()
axs[1,0].minorticks_on()
axs[1,1].minorticks_on()

#plt.tight_layout()
plt.subplots_adjust(
    wspace=0.05,   # squeeze columns together hspace=0.08,   # squeeze rows together
    hspace=0.35,
    left=0.0,
    right=1,
)

plt.savefig("../plots/final-dRdX-MASSQUERADE.pdf", bbox_inches="tight", pad_inches=0.04)
plt.show()

# fig 10

In [ ]:
Change thresho